# feature name sources — v2 / v3

**Kernel: `fttl-v2` or `fttl-v3`** (the env whose pickle you are opening).

Every place in this repo that needs "the columns the model was trained on" walks a ladder of
sources and returns the FIRST one that answers. The ladders disagree on their order:

| code | 1st | 2nd | 3rd |
|---|---|---|---|
| `features/extract_features.py` | `get_feature_names_out` | `booster.feature_names` | `feature_names_in_` |
| `src/shap_kit.py` | `booster.feature_names` | `feature_name_` | `feature_names_in_` |

Because each returns early, **nothing ever compares two sources against each other**. This
notebook asks every source separately and prints what each one says, so the ladder order can
be decided on evidence rather than on which was written first.

Nothing is written to disk.


In [ ]:
import json
import re
import sys
from pathlib import Path

import joblib
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'src' / 'config.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import config

VERSION = 'v2'          # <- set to the version whose env this kernel is
SOURCE = 'real'

print('kernel :', sys.executable)
print('env-%s :' % VERSION, config.python_bin(VERSION))
print('  ^ these two must be the same interpreter')


## 1 · Load the two pickles

The real repos pickle the estimator and the preprocessing pipeline **separately**, so the
estimator on its own has no `.steps` — which already decides part of the question: a bare
estimator makes `extract_features.py`'s first rung unreachable.


In [ ]:
model_path = config.path('model', VERSION, SOURCE)
prep_path  = config.path('preprocessor', VERSION, SOURCE)
print('model      :', model_path)
print('preprocessor:', prep_path)

est = joblib.load(model_path)
print('\nestimator   :', type(est).__name__, '| has .steps:', hasattr(est, 'steps'))
if hasattr(est, 'steps'):
    print('  steps:', [n for n, _ in est.steps])
    est = est.steps[-1][1]
    print('  -> final step:', type(est).__name__)

prep = None
try:
    prep = joblib.load(prep_path)
    print('preprocessor:', type(prep).__name__, '| has .steps:', hasattr(prep, 'steps'))
    if hasattr(prep, 'steps'):
        print('  steps:', [n for n, _ in prep.steps])
except Exception as exc:
    print('preprocessor NOT loaded:', type(exc).__name__, exc)


## 2 · Ask every source separately

`placeholder` means the names are xgboost's positional stand-ins (`f0`, `f1`, …), which happen
when the model was fit from a numpy array. Those carry an order but no names — `shap_kit_v1`
rejects them, the other three code paths accept them.


In [ ]:
PLACEHOLDER = re.compile(r'^f\d+$')

def probe(label, getter):
    try:
        names = getter()
    except Exception as exc:
        return {'source': label, 'available': False, 'n': 0,
                'placeholder': None, 'first_4': '', 'note': '%s: %s' % (type(exc).__name__, exc)}
    if names is None:
        return {'source': label, 'available': False, 'n': 0,
                'placeholder': None, 'first_4': '', 'note': 'attribute is None'}
    names = [str(n) for n in names]
    if not names:
        return {'source': label, 'available': False, 'n': 0,
                'placeholder': None, 'first_4': '', 'note': 'empty list'}
    return {'source': label, 'available': True, 'n': len(names),
            'placeholder': all(PLACEHOLDER.match(n) for n in names),
            'first_4': ', '.join(names[:4]), 'note': ''}

PROBES = [
    ('booster.feature_names',        lambda: est.get_booster().feature_names),
    ('estimator.feature_names_in_',  lambda: est.feature_names_in_),
    ('estimator.feature_name_',      lambda: est.feature_name_),          # lightgbm
    ('preprocessor.get_feature_names_out', lambda: prep.get_feature_names_out()),
    ('preprocessor[:-1].get_feature_names_out',
     lambda: prep[:-1].get_feature_names_out()),   # what extract_features.py actually calls
]

NAMES = {}
rows = []
for label, getter in PROBES:
    r = probe(label, getter)
    rows.append(r)
    if r['available']:
        NAMES[label] = [str(n) for n in getter()]

display(pd.DataFrame(rows).set_index('source'))
print('sources that answered:', list(NAMES))


## 3 · Do the sources agree?

Two questions, and they are not the same one:

* **same set** — do they name the same columns at all?
* **same order** — do they list them in the same sequence? This is the one SHAP depends on,
  because φ are attached positionally.

A pair that agrees on the set but not the order means the preprocessing head emits columns in
a different sequence from the one the booster was fit on — which would make the weaker source
an actively wrong choice for the registry.


In [ ]:
labels = list(NAMES)
if len(labels) < 2:
    print('only %d source answered — nothing to cross-check.' % len(labels))
else:
    rows = []
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            a, b = NAMES[labels[i]], NAMES[labels[j]]
            same_set = sorted(a) == sorted(b)
            rows.append({'A': labels[i], 'B': labels[j],
                         'same set': same_set,
                         'same ORDER': a == b,
                         'only in A': len(set(a) - set(b)),
                         'only in B': len(set(b) - set(a)),
                         'first mismatch at': next((k for k, (x, y) in enumerate(zip(a, b))
                                                    if x != y), None) if same_set else None})
    display(pd.DataFrame(rows))
    print('\nsame set + same ORDER  -> the ladder order does not matter; any rung is fine.')
    print('same set, DIFFERENT order -> only booster.feature_names describes what the model')
    print('                             consumed. extract_features.py must not take the head first.')


## 4 · What is actually recorded on disk

`features/registry/<v>.json` is what `shap_kit.feature_order(..., trained=...)` compares
against when the pickle cannot answer, and each attribution meta's `feature_names` is the
order the φ on disk were computed in. Both are checked here against the sources above.


In [ ]:
reg_path = config.registry_path(VERSION)
if not reg_path.exists():
    print('no registry at', reg_path)
else:
    reg = json.loads(reg_path.read_text(encoding='utf-8'))
    reg_names = [str(c) for c in (reg.get('model_features') or [])]
    print('registry :', reg_path.name)
    print('  model_features        :', len(reg_names))
    print('  model_features_source :', reg.get('model_features_source', '<< field absent — '
          'registry predates it; re-run extract_features.py >>'))
    for label, names in NAMES.items():
        print('  vs %-42s set=%-5s order=%s' % (label, sorted(names) == sorted(reg_names),
                                               names == reg_names))

shap_dir = config.path('attributions', VERSION, SOURCE,
                       split=config.SPLITS[VERSION][0]).parent
metas = sorted(shap_dir.glob('%s_attributions_*_meta.json' % VERSION))
print('\nattribution metas:', len(metas))
for m in metas:
    d = json.loads(m.read_text(encoding='utf-8'))
    phi_names = [str(c) for c in (d.get('feature_names') or [])]
    verdicts = ' '.join('%s=%s' % (lab.split('.')[-1][:14], names == phi_names)
                        for lab, names in NAMES.items())
    print('  %-46s feature_order=%-28r %s' % (m.name, d.get('feature_order'), verdicts))
